In [1]:
import os
import numpy as np
import cv2
import time
from tqdm import tqdm

from import_db import VocabNode, VocabTree, InvertedIndex, VocabTreeDB
from export_db import save_vt_model

In [2]:
# ---------- Feature Extraction ----------
def extract_sift(gray_img, nfeatures=500):
    sift = cv2.SIFT_create(nfeatures=nfeatures)
    kps, desc = sift.detectAndCompute(gray_img, None)
    if desc is None:
        return np.empty((0,128), np.float32), []
    return desc.astype(np.float32), kps
def to_rootsift(desc, eps=1e-12, l2_after=True):
    """
    Convert SIFT -> RootSIFT (Arandjelović & Zisserman, 2012).
    Steps: L1-normalize, then sqrt. Optionally L2-normalize after sqrt.
    desc: (N,128) float32
    """
    if desc is None or len(desc) == 0:
        return np.empty((0,128), np.float32)
    # L1-normalize
    desc /= (np.sum(desc, axis=1, keepdims=True) + eps)
    # element-wise sqrt
    desc = np.sqrt(desc, dtype=np.float32)
    if l2_after:
        # optional: stabilize numerics
        norms = np.linalg.norm(desc, axis=1, keepdims=True) + eps
        desc /= norms
    return desc.astype(np.float32)


In [3]:
# Hyperparameters to tune:
N_FEATURES = 2000
K = 10  # how many branches per node
L = 4  # how many levels deep
MIN_CLUSTER_SIZE = 100  # minimum points to keep splitting
MAX_ITER = 100  # max k-means iterations
STOP_PERCENT = 0.005  # drop top 0.5% most frequent leaves
STOP_FRACTION = 0.05  # drop leaves with df/N > 5%
USE_ENTROPY = True  # use entropy weighting instead of IDF

In [4]:
descriptors_list = []
images_descriptors={}
DIR_NAME='image/train/'

start_time = time.time()
for image_name in tqdm(os.listdir(DIR_NAME), desc="Extracting descriptors"):
    image = cv2.imread(f'{DIR_NAME}{image_name}', cv2.IMREAD_GRAYSCALE)
    image8bit = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')

    descriptors, keypoints = extract_sift(image8bit, nfeatures=N_FEATURES)
    descriptors = to_rootsift(descriptors)
    descriptors_list.append(descriptors)

    images_descriptors[image_name.split('.')[0]] = {
        'desc':descriptors,
        'kps':keypoints,
        'path':f'{DIR_NAME}{image_name}'
    }
    
end_time = time.time()
print(f"Descriptor extraction took {end_time - start_time:.2f} seconds")

# with open('filename.pickle', 'rb') as handle:
#     b = pickle.load(handle)

Extracting descriptors: 100%|██████████| 476/476 [00:24<00:00, 19.09it/s]

Descriptor extraction took 24.95 seconds


In [5]:
db = VocabTreeDB(
    k=K, 
    L=L, 
    min_cluster_size=MIN_CLUSTER_SIZE, 
    max_iter=MAX_ITER
)
start_time = time.time()
db.image_meta=images_descriptors
db.train(descriptors_list)
end_time = time.time()
print(f"Vocab tree training took {end_time - start_time:.2f} seconds")

Training descriptors: 822705
Vocab tree training took 117.91 seconds


In [6]:
start_time = time.time()
for image_id in tqdm(images_descriptors.keys(), desc="Adding images to index"):
    db.add_image(
        image_id, 
        descs=images_descriptors[image_id]['desc'], 
        kps=images_descriptors[image_id]['kps'], 
        path=images_descriptors[image_id]['path']
    )
end_time = time.time()
print(f"Adding images to index took {end_time - start_time:.2f} seconds")

Adding images to index: 100%|██████████| 476/476 [02:07<00:00,  3.72it/s]

Adding images to index took 127.83 seconds


In [7]:
# Compute IDF + stopwords (can be rerun after more additions)
db.finalize(
    use_entropy=USE_ENTROPY, 
    stop_percent=STOP_PERCENT, 
    stop_frac=STOP_FRACTION, 
    hard_purge=False
)

In [8]:
print("idf size:", len(db.index.idf))
print("num docs:", db.index.N)
print("sample tf:", list(db.index.doc_tf.items())[0])


idf size: 9856
num docs: 476
sample tf: (0, {2: 2, 6: 1, 8: 1, 21: 1, 27: 1, 29: 1, 59: 1, 270: 1, 275: 1, 432: 2, 433: 1, 490: 1, 547: 1, 727: 1, 748: 1, 758: 1, 924: 1, 925: 1, 964: 2, 965: 2, 966: 1, 967: 1, 972: 1, 975: 1, 981: 1, 984: 1, 996: 1, 1005: 1, 1016: 7, 1024: 7, 1025: 1, 1028: 1, 1033: 6, 1035: 2, 1037: 1, 1043: 2, 1044: 1, 1050: 2, 1051: 4, 1052: 1, 1054: 3, 1056: 1, 1058: 6, 1061: 3, 1067: 3, 1068: 1, 1069: 1, 1077: 1, 1086: 1, 1090: 1, 1108: 1, 1117: 2, 1118: 3, 1120: 9, 1121: 3, 1122: 10, 1123: 5, 1124: 1, 1125: 23, 1129: 1, 1135: 1, 1136: 1, 1138: 8, 1141: 1, 1144: 1, 1145: 14, 1147: 5, 1148: 1, 1150: 5, 1160: 1, 1162: 7, 1165: 8, 1167: 1, 1177: 1, 1186: 6, 1191: 1, 1192: 1, 1196: 2, 1198: 5, 1199: 2, 1200: 7, 1201: 2, 1202: 6, 1204: 2, 1207: 1, 1209: 4, 1213: 1, 1217: 1, 1223: 3, 1225: 2, 1226: 2, 1227: 1, 1228: 1, 1229: 3, 1230: 2, 1234: 1, 1235: 15, 1244: 1, 1266: 2, 1267: 1, 1269: 1, 1270: 1, 1273: 2, 1274: 12, 1284: 1, 1321: 1, 1322: 1, 1325: 8, 1333: 1, 1340: 

In [9]:
os.makedirs('vocab_db', exist_ok=True)
save_vt_model(db, f"vocab_db/{db.index.N}imgs-{N_FEATURES}descriptors-{K}k-{L}l-min_cluster{MIN_CLUSTER_SIZE}-max_iter{MAX_ITER}", include_meta=True)

[save_vt_model] Saved to: vocab_db/476imgs-2000descriptors-10k-4l-min_cluster100-max_iter100


In [22]:
# Query
q_descs, q_kps = extract_sift(cv2.imread('stamps/2.png', cv2.IMREAD_GRAYSCALE),nfeatures=1500)
qdesc_root = to_rootsift(q_descs)
cands = db.query(qdesc_root, topk=200)
print("Base prediction:", cands)
reranked = db.spatial_verify(qdesc_root, q_kps, cands, ratio_thresh=0.75, cap=1000, lam=0.01)
top10 = [(img, score, inlier_ratio) for img, score, inl, inlier_ratio in reranked[:10]]
print("Top 10 after RANSAC re-ranking:", top10)

sv_map = {img:(final,inl) for img,final,inl,inlier_ratio in reranked}
changes = []
for img, base in cands:
    final, inl = sv_map[img]
    changes.append((img, base, inl, final-base))
print("any inliers >", any(inl>0 for _,_,inl,_ in changes))
print("top deltas:", sorted(changes, key=lambda x: x[3], reverse=True)[:5])


Base prediction: [('213355', 0.3073907133636711), ('1699018', 0.29784086497719153), ('212705', 0.27033338544988933), ('212723', 0.2554887711262441), ('2045397', 0.24426904558059306), ('1701805', 0.23913412863678035), ('1699021', 0.20669951399627912), ('1701939', 0.1970828857373733), ('1701803', 0.19555616259299005), ('1698489', 0.18888921127638839), ('1701796', 0.18584634889855647), ('1701879', 0.1787064516263805), ('212735', 0.17445895752547713), ('1702438', 0.1731144305837223), ('1702401', 0.17183309026715082), ('1701884', 0.16894856119327542), ('1702398', 0.1651764682811262), ('1702350', 0.1627316188848053), ('212734', 0.16220967080576687), ('1701873', 0.16009562351295614), ('1702417', 0.1569026788445465), ('1702436', 0.1521995890307121), ('1701946', 0.15181980708586748), ('1702429', 0.15061810782203522), ('1701878', 0.14937665140270465), ('1701816', 0.14349377459383644), ('1701814', 0.1433920249478708), ('1701931', 0.13962558388865468), ('1702396', 0.13618380695333257), ('212715', 